# Algoritmi approssimati per trovare sottogruppi divergenti

Questo notebook segue lo stesso percorso di `DivExplorerExample.ipynb`, ma accanto a
DivExplorer (esatto e completo) usa i tre algoritmi approssimati della tesi:

1. **Random rows**: esplorazione random delle righe (matrice compatta + pazienza K);
2. **Beam search** modificata (riserva + diversità);
3. **MCTS**: Monte Carlo Tree Search con UCB.

Per ogni dataset guardiamo **quali** sottogruppi trova ogni metodo, non solo quanti.
Gli algoritmi approssimati non garantiscono di trovare tutto, ma i valori che trovano
(supporto, divergenza, t-value) sono calcolati esattamente, con le stesse formule di DivExplorer.

<a href="https://colab.research.google.com/github/TUO_UTENTE/TUO_REPO/blob/main/notebooks/ApproximateAlgorithmsExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Il notebook sta nella cartella `notebooks/` del repository: aggiungiamo la cartella
principale al path, così Python trova sia DivExplorer sia `divexplorer/approximate_algorithms/`.
Su Colab il repository non c'è: la cella lo scarica da GitHub (cambia `REPO` con il tuo).

In [ ]:
import os
import sys

REPO = "https://github.com/TUO_UTENTE/TUO_REPO.git"

if "google.colab" in sys.modules:          # su Colab: si scarica il repository
    if not os.path.exists("/content/repo"):
        os.system(f"git clone {REPO} /content/repo")
    %pip install -q -r /content/repo/requirements.txt
    ROOT = "/content/repo"
else:                                       # in locale: la cartella sopra notebooks/
    ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT)

import pandas as pd
from divexplorer import DivergenceExplorer, DivergencePatternProcessor
from divexplorer.approximate_algorithms.random_rows import random_rows
from divexplorer.approximate_algorithms.beam_search import beam_search
from divexplorer.approximate_algorithms.mcts import mcts
from divexplorer.approximate_algorithms.common import Evaluator, important_itemsets

## Funzioni di supporto

- `run_all` lancia i quattro metodi sugli stessi dati. Beam search e MCTS ricevono
  lo **stesso tempo** impiegato da random rows.
- `for_processor` aggiunge al risultato le colonne che servono a `DivergencePatternProcessor`
  (`support_count` e la riga dell'insieme vuoto), così possiamo usare redundancy pruning
  e Shapley value anche sui risultati degli algoritmi approssimati.
- `found_by` dice, per ogni sottogruppo di una lista, quali metodi lo hanno trovato.

In [ ]:
def for_processor(res, n):
    """Aggiunge support_count e la riga dell'insieme vuoto (divergenza 0),
    come nell'output di DivExplorer."""
    res = res.copy()
    res["support_count"] = (res["support"] * n).round().astype(int)
    if not (res["length"] == 0).any():
        empty = {c: 0.0 for c in res.columns}
        empty.update({"support": 1.0, "itemset": frozenset(), "length": 0, "support_count": n})
        res = pd.concat([pd.DataFrame([empty]), res], ignore_index=True)
    return res


def run_all(df, attributes, outcome, kind, min_support, K=2000, seed=0):
    """kind = "bool" (boolean_outcomes) oppure "num" (quantitative_outcomes)."""
    n = len(df)
    results, info = {}, {}

    explorer = DivergenceExplorer(df)
    args = {"boolean_outcomes": [outcome]} if kind == "bool" else {"quantitative_outcomes": [outcome]}
    results["DivExplorer"] = explorer.get_pattern_divergence(
        min_support=min_support, attributes=attributes, **args)

    res, ev, inf = random_rows(df, attributes, outcome, kind, min_support, K=K, seed=seed)
    results["Random rows"], info["Random rows"] = for_processor(res, n), inf
    budget = inf["tempo"]

    res, ev, inf = beam_search(df, attributes, outcome, kind, min_support, time_budget=budget)
    results["Beam search"], info["Beam search"] = for_processor(res, n), inf

    res, ev, inf = mcts(df, attributes, outcome, kind, min_support, time_budget=budget, seed=seed)
    results["MCTS"], info["MCTS"] = for_processor(res, n), inf

    for name, r in results.items():
        print(f"{name:12s}: {(r['length'] > 0).sum()} sottogruppi frequenti",
              "" if name == "DivExplorer" else f"  ({info[name]})")
    return results


def top(res, outcome, k=10, ascending=False):
    """I k sottogruppi con divergenza più alta (o più bassa con ascending=True)."""
    cols = ["support", "itemset", outcome, f"{outcome}_div", f"{outcome}_t", "length", "support_count"]
    return (res[res["length"] > 0].sort_values(f"{outcome}_div", ascending=ascending)
            [cols].head(k).reset_index(drop=True))


def found_by(results, itemsets, outcome):
    """Per ogni sottogruppo: divergenza (da DivExplorer) e quali metodi lo hanno trovato."""
    ref = dict(zip(results["DivExplorer"]["itemset"], results["DivExplorer"][f"{outcome}_div"]))
    rows = []
    for it in itemsets:
        row = {"itemset": it, f"{outcome}_div": ref[it]}
        for name, r in results.items():
            row[name] = "✓" if it in set(r["itemset"]) else "✗"
        rows.append(row)
    return pd.DataFrame(rows)

## Un esempio con il census

### Outcome numerico (quantitativo)

Come nel notebook di DivExplorer: stato, sesso, istruzione, età e reddito. Cerchiamo i
sottogruppi con reddito medio più alto e più basso. Il reddito (`PTOTVAL`) è l'*outcome
quantitativo*.

In [ ]:
df_census = pd.read_csv(os.path.join(ROOT, "datasets", "census_income.csv"))
df_census["AGE_RANGE"] = 10 * (df_census["A_AGE"] // 10)
df_census.head()

Stessi parametri dell'esempio originale: attributi `SEX`, `EDUCATION`, `AGE_RANGE` e
supporto minimo 0.001 (almeno lo 0,1% delle persone).

In [ ]:
census = run_all(df_census, attributes=["SEX", "EDUCATION", "AGE_RANGE"],
                 outcome="PTOTVAL", kind="num", min_support=0.001)

### I sottogruppi trovati da ogni metodo

Le tabelle hanno le stesse colonne di DivExplorer: `support`, `itemset`, `PTOTVAL`
(reddito medio nel sottogruppo), `PTOTVAL_div` (differenza dal reddito medio del dataset),
`PTOTVAL_t` (t-value), `length`, `support_count`.

In [ ]:
top(census["DivExplorer"], "PTOTVAL")

In [ ]:
top(census["Random rows"], "PTOTVAL")

In [ ]:
top(census["Beam search"], "PTOTVAL")

In [ ]:
top(census["MCTS"], "PTOTVAL")

### Chi ha trovato cosa

I 10 sottogruppi con reddito più **alto** e i 10 con reddito più **basso** secondo
DivExplorer: per ognuno, quali metodi lo hanno trovato.

In [ ]:
best = list(top(census["DivExplorer"], "PTOTVAL")["itemset"])
worst = list(top(census["DivExplorer"], "PTOTVAL", ascending=True)["itemset"])
found_by(census, best + worst, "PTOTVAL")

### Redundancy pruning

Come nell'esempio originale, togliamo gli item che cambiano la divergenza di meno di 20.000.
Lo facciamo su ogni risultato con `DivergencePatternProcessor`.

In [ ]:
for name, res in census.items():
    pruned = DivergencePatternProcessor(res, "PTOTVAL").redundancy_pruning(th_redundancy=20000)
    print(f"{name:12s}: {(pruned['length'] > 0).sum()} sottogruppi dopo il pruning")

DivergencePatternProcessor(census["Random rows"], "PTOTVAL").redundancy_pruning(
    th_redundancy=20000).sort_values("PTOTVAL_div", ascending=False).head(10)

### Shapley value

Prendiamo il sottogruppo più divergente e calcoliamo il contributo di ogni item.
Per lo Shapley value servono **tutti i sottoinsiemi** del sottogruppo: random rows li ha
sempre (ogni campione calcola tutti i sottoinsiemi), quindi il valore è esatto e identico a
DivExplorer. Beam search e MCTS non danno questa garanzia.

In [ ]:
pattern = top(census["DivExplorer"], "PTOTVAL")["itemset"].iloc[0]
print(pattern)
pd.DataFrame({name: DivergencePatternProcessor(census[name], "PTOTVAL").shapley_value(pattern)
              for name in ["DivExplorer", "Random rows"]})

In [ ]:
DivergencePatternProcessor(census["Random rows"], "PTOTVAL").plot_shapley_value(
    pattern=pattern, figsize=(2, 2), title="Contributo degli item (random rows)")

## Divergenza in un classificatore: COMPAS

### False positive rate

Come nell'esempio originale: un falso positivo è un imputato per cui il modello prevede
recidiva ma che non recidiva. L'outcome `fp` vale 1 (falso positivo), 0 (vero negativo)
o NaN (classe positiva, esclusa dalla media). È un *outcome booleano*.

In [ ]:
from divexplorer.outcomes import get_false_positive_rate_outcome

compas_df = pd.read_csv(os.path.join(ROOT, "datasets", "compas_discretized.csv"))
compas_df["fp"] = get_false_positive_rate_outcome(compas_df["class"], compas_df["predicted"])
compas_df = compas_df[["race", "#prior", "sex", "age", "fp"]]
compas_df.head()

In [ ]:
compas = run_all(compas_df, attributes=["race", "#prior", "sex", "age"],
                 outcome="fp", kind="bool", min_support=0.1)

In [ ]:
top(compas["DivExplorer"], "fp")

In [ ]:
top(compas["Random rows"], "fp")

In [ ]:
top(compas["Beam search"], "fp")

In [ ]:
top(compas["MCTS"], "fp")

### Chi ha trovato cosa

I sottogruppi **importanti** (T_min): frequenti, statisticamente significativi (t ≥ 2) e
non ridondanti (ε = 0.05). Sono quelli che confrontiamo nella tabella dei risultati.

In [ ]:
ev = Evaluator(compas_df, ["race", "#prior", "sex", "age"], "fp", "bool")
T_min = important_itemsets(compas["DivExplorer"], "fp", tau=2.0, eps=0.05, ev=ev)
found_by(compas, T_min, "fp").sort_values("fp_div", ascending=False, ignore_index=True)

### Shapley value (locale)

In [ ]:
pattern = top(compas["DivExplorer"], "fp")["itemset"].iloc[0]
print(pattern)
pd.DataFrame({name: DivergencePatternProcessor(compas[name], "fp").shapley_value(pattern)
              for name in ["DivExplorer", "Random rows"]})

In [ ]:
DivergencePatternProcessor(compas["Random rows"], "fp").plot_shapley_value(
    pattern=pattern, figsize=(2, 2), title="Contributo degli item (random rows)")

### Redundancy pruning

In [ ]:
for name, res in compas.items():
    pruned = DivergencePatternProcessor(res, "fp").redundancy_pruning(th_redundancy=0.01)
    print(f"{name:12s}: {(pruned['length'] > 0).sum()} sottogruppi dopo il pruning")

DivergencePatternProcessor(compas["Random rows"], "fp").redundancy_pruning(
    th_redundancy=0.01).sort_values("fp_div", ascending=False).head(5)

### Global Shapley value

Il global Shapley value usa **tutti** i sottogruppi frequenti. È esatto solo se il metodo
li ha trovati tutti: con un algoritmo approssimato è un'approssimazione. Qui, su dati
piccoli, i valori dovrebbero coincidere.

In [ ]:
gsv = pd.DataFrame({name: DivergencePatternProcessor(res, "fp").global_shapley_value()
                    for name, res in compas.items()})
gsv.sort_values("DivExplorer")

In [ ]:
DivergencePatternProcessor(compas["Random rows"], "fp").plot_shapley_value(
    shapley_values=gsv["Random rows"].to_dict(), figsize=(3, 4),
    title="Global Shapley value (random rows)")